# 11 — A national transport model: Great Britain

This notebook builds and runs a **national strategic model of Great Britain** from
scratch: the real motorway and trunk-road network, the 75 largest cities and
strategic towns as zones, gravity demand from population, and a national
equilibrium assignment — finishing with the busiest corridors in the country and
the flows crossing the England–Scotland border.

Everything the notebook needs ships in the `data/` folder next to it:

| file | contents | source |
|---|---|---|
| `uk_strategic_roads.geojson.gz` | 64,733 links / ~40,000 km of GB motorways, trunk roads and ramps, topologically connected | © OpenStreetMap contributors (ODbL), extracted via Overpass API |
| `uk_cities.csv` | 75 cities and strategic towns with coordinates and population | assembled from public population estimates |
| `gb_boundary.geojson` | simplified GB outline for the maps | public-domain world boundaries |

No downloads happen at run time — the model is fully self-contained.

In [1]:
import gzip
import json
import time
import warnings
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import LineString, shape

warnings.filterwarnings("ignore")

DATA = Path("data")
with gzip.open(DATA / "uk_strategic_roads.geojson.gz", "rt", encoding="utf-8") as fh:
    roads_gj = json.load(fh)
cities = pd.read_csv(DATA / "uk_cities.csv")
boundary = gpd.GeoDataFrame(geometry=[shape(f["geometry"]) for f in json.load(open(DATA / "gb_boundary.geojson"))["features"]], crs=4326)

roads = gpd.GeoDataFrame(
    [{"cls": f["properties"]["class"], "ref": f["properties"]["ref"]} for f in roads_gj["features"]],
    geometry=[shape(f["geometry"]) for f in roads_gj["features"]], crs=4326)

km = roads.to_crs(27700).length.sum() / 1000
print(f"{len(roads):,} links / {km:,.0f} km of strategic roads; "
      f"{len(cities)} cities, {cities.population.sum() / 1e6:.1f}M residents")
roads.groupby("cls").size()

64,733 links / 37,206 km of strategic roads; 74 cities, 34.7M residents


cls
motorway     3172
ramp        13862
trunk       47699
dtype: int64

In [2]:
# JupyterGIS map helper ------------------------------------------------------
# GISDocument is JupyterGIS' notebook API: it builds a live, QGIS-like map
# document rendered directly in JupyterLab. Layers added from GeoDataFrames
# are converted to GeoJSON on the fly.
import json
from jupytergis import GISDocument

OSM_TILES = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"

def new_map(gdf_for_extent=None, zoom=12):
    """Create a GISDocument centred on a layer, with an OpenStreetMap basemap."""
    kwargs = {}
    if gdf_for_extent is not None:
        b = gdf_for_extent.total_bounds  # (minx, miny, maxx, maxy)
        kwargs = {"longitude": (b[0] + b[2]) / 2, "latitude": (b[1] + b[3]) / 2, "zoom": zoom}
    doc = GISDocument(**kwargs)
    doc.add_raster_layer(OSM_TILES, name="OpenStreetMap", attribution="(C) OpenStreetMap contributors", opacity=0.6)
    return doc

def add_gdf(doc, gdf, name, **kwargs):
    """Add a GeoDataFrame to the map as a GeoJSON layer."""
    return doc.add_geojson_layer(data=json.loads(gdf.to_json()), name=name, **kwargs)

## The raw ingredients

The strategic road network, the GB outline and the 75 model zones.

In [3]:
from jupytergis_lab.notebook.symbology import constant

cities_gdf = gpd.GeoDataFrame(cities, geometry=gpd.points_from_xy(cities.lon, cities.lat), crs=4326)

display = roads[roads.cls != "ramp"].copy()
display["geometry"] = display.geometry.simplify(0.01)

doc = new_map(boundary, zoom=6)
add_gdf(doc, boundary, "Great Britain", opacity=0.2, symbology=[[constant("#94a3b8").encoding("fill")]])
add_gdf(doc, display[display.cls == "trunk"][["ref", "geometry"]], "trunk roads",
        symbology=[[constant("#64748b").encoding("stroke")]])
add_gdf(doc, display[display.cls == "motorway"][["ref", "geometry"]], "motorways",
        symbology=[[constant("#1d4ed8").encoding("stroke")]])
add_gdf(doc, cities_gdf[["city", "population", "geometry"]], "cities",
        symbology=[[constant("#dc2626").encoding("fill")]])
doc

## 1. Building the national network

Every link is inserted through AequilibraE's standard editing path, so the
database consistency triggers create the nodes and maintain `a_node`/`b_node`
and lengths — 64,000 links, exactly as if they had been digitised by hand.
Class-based speeds and per-direction capacities give the free-flow attributes.

In [4]:
from aequilibrae.project import Project

SPEC = {  # free-flow speed km/h, capacity veh/h/direction
    "motorway": (105, 4000),
    "trunk": (75, 1800),
    "ramp": (55, 1500),
}

fldr = str(Path(gettempdir()) / uuid4().hex)
project = Project()
project.new(fldr)

t0 = time.perf_counter()
with project.db_connection as conn:
    for lt, (speed, cap) in SPEC.items():
        conn.execute("insert into link_types (link_type, link_type_id, description, speed) values (?,?,?,?)",
                     (lt, lt[0], f"{lt} (UK strategic network)", speed))
    lid = 0
    for f in roads_gj["features"]:
        cls = f["properties"]["class"]
        coords = f["geometry"]["coordinates"]
        if coords[0] == coords[-1]:
            continue  # degenerate rings cannot carry through traffic
        speed, cap = SPEC[cls]
        lid += 1
        wkt = "LINESTRING(" + ", ".join(f"{x} {y}" for x, y in coords) + ")"
        conn.execute(
            "insert into links (link_id, a_node, b_node, link_type, modes, direction, "
            " speed_ab, speed_ba, capacity_ab, capacity_ba, name, geometry) "
            "values (?, 0, 0, ?, 'c', 0, ?, ?, ?, ?, ?, GeomFromText(?, 4326))",
            (lid, cls, speed, speed, cap, cap, f["properties"]["ref"], wkt))
    conn.execute("update links set travel_time_ab = distance / 1000.0 / speed_ab * 60, "
                 "travel_time_ba = distance / 1000.0 / speed_ba * 60")
    conn.commit()
    n_nodes = conn.execute("select count(*) from nodes").fetchone()[0]
print(f"inserted {lid:,} links in {time.perf_counter() - t0:.0f}s; "
      f"triggers created {n_nodes:,} nodes")

inserted 64,730 links in 20s; triggers created 46,438 nodes


## 2. Cities become zones

Each city is attached to its nearest network node by a centroid connector, and
the city node is flagged as a zone centroid.

In [5]:
from scipy.spatial import cKDTree

nodes_df = project.network.nodes.data
xy = np.c_[nodes_df.geometry.x * np.cos(np.radians(54.5)), nodes_df.geometry.y]
tree = cKDTree(xy)

with project.db_connection as conn:
    for i, c in cities.iterrows():
        _, j = tree.query([c.lon * np.cos(np.radians(54.5)), c.lat])
        nearest = nodes_df.geometry.iloc[j]
        lid += 1
        wkt = f"LINESTRING({c.lon} {c.lat}, {nearest.x} {nearest.y})"
        conn.execute(
            "insert into links (link_id, a_node, b_node, link_type, modes, direction, "
            " speed_ab, speed_ba, capacity_ab, capacity_ba, name, geometry) "
            "values (?, 0, 0, 'centroid_connector', 'c', 0, 48, 48, 10000, 10000, ?, GeomFromText(?, 4326))",
            (lid, f"{c.city} connector", wkt))
    conn.execute("update links set travel_time_ab = distance / 1000.0 / speed_ab * 60, "
                 "travel_time_ba = distance / 1000.0 / speed_ba * 60 where travel_time_ab is null")
    conn.commit()

nodes_df = project.network.nodes.data  # refresh: connector end nodes now exist
key = (nodes_df.geometry.x.round(5).astype(str) + "|" + nodes_df.geometry.y.round(5).astype(str))
lookup = dict(zip(key, nodes_df.node_id))
cities["node_id"] = [lookup[f"{round(c.lon, 5)}|{round(c.lat, 5)}"] for c in cities.itertuples()]

with project.db_connection as conn:
    conn.executemany("update nodes set is_centroid = 1 where node_id = ?",
                     [(int(n),) for n in cities.node_id])
    conn.commit()
print(f"{len(cities)} centroids connected (e.g. {cities.city.iloc[0]} -> node {cities.node_id.iloc[0]})")

74 centroids connected (e.g. London -> node 46439)


## 3. National skims

Free-flow drive times between all 75 cities.

In [6]:
from aequilibrae.paths import NetworkSkimming

project.network.build_graphs(modes=["c"])
graph = project.network.graphs["c"]
graph.set_graph("travel_time")
graph.set_skimming(["travel_time", "distance"])
graph.set_blocked_centroid_flows(True)

t0 = time.perf_counter()
skimmer = NetworkSkimming(graph)
skimmer.execute()
tt = np.array(skimmer.results.skims.get_matrix("travel_time"), copy=True)
print(f"skimmed {tt.shape[0]}x{tt.shape[1]} city pairs in {time.perf_counter() - t0:.1f}s")

by_node = cities.set_index("node_id").reindex(graph.centroids)
times = pd.DataFrame(tt, index=by_node.city, columns=by_node.city)
pairs = [("London", "Birmingham"), ("London", "Manchester"), ("London", "Edinburgh"),
         ("Manchester", "Glasgow"), ("Bristol", "Newcastle upon Tyne"), ("Cardiff", "Norwich")]
pd.DataFrame([{"from": a, "to": b, "free-flow drive": f"{times.loc[a, b] / 60:.1f} h"} for a, b in pairs])

                                                  :   0%|          | 0/74 [00:00<?, ?it/s]

skimmed 74x74 city pairs in 0.1s


,from,to,free-flow drive
0,London,Birmingham,1.9 h
1,London,Manchester,3.2 h
2,London,Edinburgh,6.5 h
3,Manchester,Glasgow,3.4 h
4,Bristol,Newcastle upon Tyne,4.7 h
5,Cardiff,Norwich,4.8 h


## 4. National demand

Gravity demand between cities — productions from population, attractions from a
sublinear population proxy, a slow exponential decay suited to long-distance
travel, balanced with doubly-constrained IPF.

In [7]:
pop = by_node.population.to_numpy(dtype=float)
productions = pop * 0.05                      # peak-hour strategic trips per resident
attractions = np.power(pop, 0.9)
attractions *= productions.sum() / attractions.sum()

imp = tt.copy()
np.fill_diagonal(imp, np.nan)
imp[~np.isfinite(imp)] = np.nan

T = np.outer(productions, attractions) * np.exp(-0.02 * np.nan_to_num(imp, nan=1e4))
T[np.isnan(imp)] = 0.0
np.fill_diagonal(T, 0.0)
for it in range(100):
    rs = T.sum(1); T *= np.divide(productions, rs, out=np.zeros_like(rs), where=rs > 0)[:, None]
    cs = T.sum(0); T *= np.divide(attractions, cs, out=np.zeros_like(cs), where=cs > 0)[None, :]
    gap = np.abs(T.sum(1) - productions).sum() / productions.sum()
    if gap < 1e-5:
        break
print(f"IPF: {it + 1} iterations; {T.sum():,.0f} inter-city trips in the peak hour")
biggest = np.unravel_index(np.argmax(T), T.shape)
print(f"largest movement: {by_node.city.iloc[biggest[0]]} -> {by_node.city.iloc[biggest[1]]} "
      f"({T[biggest]:,.0f} trips)")

IPF: 22 iterations; 1,736,060 inter-city trips in the peak hour
largest movement: London -> Birmingham (65,495 trips)


## 5. National equilibrium assignment

In [8]:
from aequilibrae.matrix import AequilibraeMatrix
from aequilibrae.paths import TrafficAssignment, TrafficClass

demand = AequilibraeMatrix()
demand.create_empty(zones=graph.num_zones, matrix_names=["matrix"], memory_only=True)
demand.index = graph.centroids[:]
demand.matrices[:, :, 0] = T
demand.computational_view()

assig = TrafficAssignment()
assig.add_class(TrafficClass(name="car", graph=graph, matrix=demand))
assig.set_vdf("BPR")
assig.set_vdf_parameters({"alpha": 0.15, "beta": 4.0})
assig.set_capacity_field("capacity")
assig.set_time_field("travel_time")
assig.set_algorithm("bfw")
assig.max_iter = 30
assig.rgap_target = 0.001
t0 = time.perf_counter()
assig.execute()
print(f"equilibrium in {time.perf_counter() - t0:.0f}s")

car                                               :   0%|          | 0/74 [00:00<?, ?it/s]

Equilibrium Assignment                            :   0%|          | 0/30 [00:00<?, ?it/s]

equilibrium in 4s


## 6. The national flow map

In [9]:
from jupytergis_lab.notebook.symbology import field

res = assig.results()
links_gdf = project.network.links.data
loaded = links_gdf.merge(res.reset_index(), on="link_id")
loaded = loaded[(loaded.matrix_tot > 200) & (loaded.link_type != "centroid_connector")].copy()
loaded["flow"] = loaded.matrix_tot.round(0)
loaded["geometry"] = loaded.geometry.simplify(0.005)

doc = new_map(boundary, zoom=6)
add_gdf(doc, boundary, "Great Britain", opacity=0.2, symbology=[[constant("#94a3b8").encoding("fill")]])
add_gdf(doc, loaded[["link_id", "flow", "geometry"]], "assigned flows",
        symbology=[[field("flow").colormap("YlOrRd", domain=(0.0, float(loaded.flow.max()))).encoding("stroke")]])
add_gdf(doc, cities_gdf[["city", "geometry"]], "cities", symbology=[[constant("#111827").encoding("fill")]])
doc

The busiest strategic corridors, ranked by vehicle-kilometres:

In [10]:
loaded["veh_km"] = loaded["matrix_tot"] * loaded["distance"] / 1000
corridors = (loaded[loaded["name"].str.len() > 0].groupby("name")
             .agg(veh_km=("veh_km", "sum"), peak_flow=("matrix_tot", "max"))
             .nlargest(10, "veh_km").round(0).astype(int))
corridors

,veh_km,peak_flow
name,,
M1,19924828,64837
M4,17540804,64883
M6,16954426,37041
M25,14793769,58602
M40,11725841,48317
A1(M),11444399,53539
M5,11053865,33404
A1,8925607,56513
M3,7527634,59056


## 7. The England–Scotland border screenline

In [11]:
border = LineString([(-3.6, 54.98), (-1.8, 55.82)])
crossing = loaded[loaded.geometry.intersects(border)]
print(f"{len(crossing)} strategic links cross the border screenline; "
      f"{crossing.matrix_tot.sum():,.0f} vehicles/h in the modeled peak:")
crossing[["name", "link_type", "matrix_tot"]].sort_values("matrix_tot", ascending=False) \
        .rename(columns={"matrix_tot": "flow"}).round(0).reset_index(drop=True)

6 strategic links cross the border screenline; 91,858 vehicles/h in the modeled peak:


,name,link_type,flow
0,A74(M),motorway,26169.0
1,A74(M),motorway,25943.0
2,A75,trunk,11643.0
3,A1,trunk,9820.0
4,A68,trunk,9514.0
5,A7,trunk,8769.0


## Wrap-up, provenance and caveats

A national model, end to end: 64,000 real strategic-road links inserted through
the consistency triggers, 75 population-weighted zones, free-flow national skims,
gravity + IPF demand, a converged BPR equilibrium, corridor rankings and a border
screenline — all from three small files in `data/`.

**Data provenance.** The road network is © OpenStreetMap contributors, licensed
ODbL, extracted from the Overpass API (motorway, trunk and their ramps; merged,
topologically noded, largest connected component) in August 2026. The GB outline
is from public-domain world boundary data. City populations are approximate
urban-area figures assembled from public estimates.

**Caveats.** This is a teaching model: demand rates, the deterrence parameter and
capacities are plausible but uncalibrated; the network carries no local roads, so
urban flows concentrate on the strategic network; and free-flow times ignore
junction delay. Calibrating those against observed data is exactly the workflow
notebooks 04–08 cover.